In [4]:
# ── link() — create symlinks in train/ and test/ without moving originals ─────
#
# Test  = videos listed in the "Foundational Model" sheet of the Excel
# Train = everything else
#
# Destination layout (symlinks → originals):
#   FOUND_FRAMES/train/{dataset}/{video_folder} → ../../{dataset}/{video_folder}
#   FOUND_FRAMES/test/ {dataset}/{video_folder} → ../../{dataset}/{video_folder}
#
# Originals are NEVER touched.

import re
import os
import pandas as pd
from pathlib import Path

FOUND_FRAMES = Path('/media/HDD1/moritz/Extracted Frames')
EXCEL_PATH = Path(
    'final split.xlsx')

# Maps "Dataset" column in the Foundational Model sheet → disk sub-folder(s)
DATASET_MAP = {
    'ATLR':              ['ATLR'],
    'MVD':               ['MVD'],
    'Tumour_Resections': ['Tumour_Resections'],
    'VS':                ['VS_Retrosigmoid', 'VS_Translabyrinthine'],
    'Aneurysm_Clipping': ['Aneurysm_Clipping'],
}

# Hardcoded equivalences: these specific IDs are treated as the same video.
_EQUIVALENCES = [
    {'RS-046', 'TL-046'},
]


def _normalize(s: str) -> str:
    return str(s).replace(', ', '_').replace('/', '').replace(' ', '_')


def _equivalent(a: str, b: str) -> bool:
    """Return True if a and b are in the same hardcoded equivalence group."""
    for group in _EQUIVALENCES:
        if any(a.startswith(g) for g in group) and any(b.startswith(g) for g in group):
            return True
    return False


def _match(excel_name: str, folder_name: str) -> bool:
    if excel_name == folder_name:
        return True
    if folder_name.startswith(excel_name + '_') or folder_name == excel_name:
        return True
    if _normalize(excel_name) == folder_name:
        return True
    if _equivalent(excel_name, folder_name):
        return True
    return False


def link():
    # ── 1. Read test list from Excel ─────────────────────────────────────────
    xl = pd.ExcelFile(EXCEL_PATH)
    df = xl.parse('Foundational Model', header=None)
    df.columns = ['dataset', 'name', 'link']
    df = df.dropna(subset=['dataset', 'name'])
    df = df[df['dataset'] != 'Dataset']

    # ── 2. Scan disk, classify every video folder ─────────────────────────────
    test_set = set()
    unmatched = []

    for _, row in df.iterrows():
        excel_ds = str(row['dataset']).strip()
        excel_name = str(row['name']).strip()
        disk_ds_list = DATASET_MAP.get(excel_ds, [excel_ds])

        found = False
        for disk_ds in disk_ds_list:
            ds_path = FOUND_FRAMES / disk_ds
            if not ds_path.exists():
                continue
            for folder in sorted(ds_path.iterdir()):
                if folder.is_dir() and _match(excel_name, folder.name):
                    test_set.add((disk_ds, folder.name))
                    found = True
        if not found:
            unmatched.append((excel_ds, excel_name))

    # all video folders currently on disk (skip train/ and test/ if they exist)
    all_videos = []
    for ds_path in sorted(FOUND_FRAMES.iterdir()):
        if ds_path.name in ('train', 'test') or not ds_path.is_dir():
            continue
        for folder in sorted(ds_path.iterdir()):
            if folder.is_dir():
                all_videos.append((ds_path.name, folder.name))

    train_set = [(ds, v) for (ds, v) in all_videos if (ds, v) not in test_set]

    # ── 3. Print plan ─────────────────────────────────────────────────────────
    sep = '─' * 64
    print(sep)
    print(f'  TRAIN : {len(train_set):>3} videos')
    print(f'  TEST  : {len(test_set):>3} videos')
    print(sep)

    print(f'\n✅  TEST videos ({len(test_set)}):')
    for ds, v in sorted(test_set):
        print(f'    {ds}/{v}')

    if unmatched:
        print(f'\n⚠️   Excel entries NOT found on disk ({len(unmatched)}):')
        for excel_ds, name in unmatched:
            print(f'    [{excel_ds}]  {name}')
    else:
        print('\n✅  All Excel entries matched on disk.')

    print(f'\nWill create symlinks under:')
    print(f'    {FOUND_FRAMES}/train/{{dataset}}/{{video}} → original')
    print(f'    {FOUND_FRAMES}/test/ {{dataset}}/{{video}} → original')
    print(f'\nOriginals will NOT be moved or modified.')

    # ── 4. Confirm ────────────────────────────────────────────────────────────
    ans = input(
        '\nType  yes  to create symlinks, anything else to abort: ').strip().lower()
    if ans != 'yes':
        print('Aborted — nothing changed.')
        return

    # ── 5. Create symlinks ────────────────────────────────────────────────────
    linked_train = linked_test = 0

    for ds, v in train_set:
        src = FOUND_FRAMES / ds / v          # original (absolute)
        dst = FOUND_FRAMES / 'train' / ds / v
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists() and not dst.is_symlink():
            os.symlink(src, dst)
        linked_train += 1
        if linked_train % 10 == 0:
            print(f'  train … {linked_train}/{len(train_set)}')

    for ds, v in sorted(test_set):
        src = FOUND_FRAMES / ds / v
        dst = FOUND_FRAMES / 'test' / ds / v
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists() and not dst.is_symlink():
            os.symlink(src, dst)
        linked_test += 1

    print(f'\n✅  Done.  Linked {linked_train} → train/  |  {linked_test} → test/')
    print(f'   Originals untouched at {FOUND_FRAMES}/{{dataset}}/{{video}}')


link()


────────────────────────────────────────────────────────────────
  TRAIN : 109 videos
  TEST  :  29 videos
────────────────────────────────────────────────────────────────

✅  TEST videos (29):
    ATLR/ATLR_16
    ATLR/ATLR_29
    ATLR/ATLR_31
    ATLR/ATLR_34
    ATLR/ATLR_5
    Aneurysm_Clipping/ACOM_left_pterional_approach_05032021_QSC93529_Volume_4_Video_4
    Aneurysm_Clipping/ACOM_right_pterional_approach_03022025_21428304_8._Raafay_R_Pterional_for_AComm_Aneurysm_-_MRN_-_21428304
    Aneurysm_Clipping/ACOM_right_pterional_approach_15112024_40704288_3._Raafay_R_Pterional_for_AComm_Aneurysm_-_MRN_-_40704288
    Aneurysm_Clipping/ICA_left_pterional_approach_15102021_0.3018731_ICA_Volume_1_Video_2
    Aneurysm_Clipping/MCA_left_pterional_approach_14012025_21905025_7._Raafay_L_MCA_Aneurysm_-_MRN_-_21905025
    Aneurysm_Clipping/MCA_left_pterional_approach_20062025_22012130_17._Raafay_L_MCA_Aneurysm_-_MRN_-_22012130
    Aneurysm_Clipping/MCA_left_pterional_approach_2482021_21144934_MC

  train … 10/109
  train … 20/109
  train … 30/109
  train … 40/109
  train … 50/109
  train … 60/109
  train … 70/109
  train … 80/109
  train … 90/109
  train … 100/109

✅  Done.  Linked 109 → train/  |  29 → test/
   Originals untouched at /media/HDD1/moritz/Extracted Frames/{dataset}/{video}


In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('df_all.csv')

df


,video_idx,Image,Phase,Grasper,Bipolar,Hook,Scissors,Clipper,Irrigator,SpecimenBag,...,phase_num,time_til_end_phase_0,time_til_end_phase_1,time_til_end_phase_2,time_til_end_phase_3,time_til_end_phase_4,time_til_end_phase_5,time_til_end_phase_6,time_til_next_phase,split
0,1,1,Preparation,1,0,0,0,0,0,0,...,0,21,673,887,1470,1568,1641,1733,21,train
1,1,2,Preparation,1,0,0,0,0,0,0,...,0,20,672,886,1469,1567,1640,1732,20,train
2,1,3,Preparation,1,0,0,0,0,0,0,...,0,19,671,885,1468,1566,1639,1731,19,train
3,1,4,Preparation,1,0,0,0,0,0,0,...,0,18,670,884,1467,1565,1638,1730,18,train
4,1,5,Preparation,0,0,0,0,0,0,0,...,0,17,669,883,1466,1564,1637,1729,17,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184493,80,1720,GallbladderRetraction,1,0,0,0,0,0,0,...,6,-1662,-583,-451,-188,-142,-120,5,5,train
184494,80,1721,GallbladderRetraction,1,0,0,0,0,0,0,...,6,-1663,-584,-452,-189,-143,-121,4,4,train
184495,80,1722,GallbladderRetraction,1,0,0,0,0,0,0,...,6,-1664,-585,-453,-190,-144,-122,3,3,train
184496,80,1723,GallbladderRetraction,1,0,0,0,0,0,0,...,6,-1665,-586,-454,-191,-145,-123,2,2,train


In [3]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import torch
import numpy as np
from transformers import ConvNextImageProcessor, ConvNextModel

# from splitter.py
FRAMES_DIR = Path('/media/HDD1/moritz/AI for Surg/frames_224')
EMBEDDINGS_DIR = Path('Convenxt_cholec_embeddings')
MANIFEST_PATH = EMBEDDINGS_DIR / 'embedding_manifest.csv'
CONVNEXT_MODEL_NAME = 'facebook/convnext-large-224-22k-1k'
CONVNEXT_CHECKPOINT = None  # e.g. Path('/path/to/checkpoint.pt')
BATCH_SIZE = 48  # matches splitter.py default
FORCE_RECOMPUTE = False
FAIL_ON_MISSING = True

if not FRAMES_DIR.exists():
    raise FileNotFoundError(f'Missing frames dir: {FRAMES_DIR}')

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Deterministic mapping for this dataset:
# frame file = frames_224/videoXX/videoXX_000001.png
video_num = df['video_idx'].astype(int)
frame_num = df['Image'].astype(int)
df['video_dir'] = video_num.map(lambda v: f'video{v:02d}')
df['frame_file'] = [f'{vd}_{fn:06d}.png' for vd,
                    fn in zip(df['video_dir'], frame_num)]
df['frame_path'] = [str(FRAMES_DIR / vd / ff)
                    for vd, ff in zip(df['video_dir'], df['frame_file'])]
df['embedding_path'] = [
    str(EMBEDDINGS_DIR / vd / Path(ff).with_suffix('.npy')) for vd, ff in zip(df['video_dir'], df['frame_file'])
]

exists_mask = df['frame_path'].map(lambda p: Path(p).exists())
missing_count = int((~exists_mask).sum())
print(f'Mapped {len(df)} rows to frame paths. Missing: {missing_count}')
if missing_count > 0 and FAIL_ON_MISSING:
    missing_examples = df.loc[~exists_mask, 'frame_path'].head(5).tolist()
    raise RuntimeError(
        f'Missing frame files (showing up to 5): {missing_examples}')

# Build worklist from unique (frame_path, embedding_path) pairs.
# This guarantees exactly one .npy embedding per source frame.
pairs = df.loc[exists_mask, ['frame_path', 'embedding_path']].drop_duplicates()
if FORCE_RECOMPUTE:
    todo = pairs
else:
    todo = pairs[~pairs['embedding_path'].map(lambda p: Path(p).exists())]

print(
    f'Unique frame embeddings to write: {len(todo)} (total unique frames: {len(pairs)})')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

processor = ConvNextImageProcessor.from_pretrained(
    CONVNEXT_MODEL_NAME,
    size={'shortest_edge': 224},
    crop_size={'height': 224, 'width': 224},
)
model = ConvNextModel.from_pretrained(CONVNEXT_MODEL_NAME)

if CONVNEXT_CHECKPOINT is not None:
    checkpoint = torch.load(CONVNEXT_CHECKPOINT, map_location=device)
    state_dict = checkpoint['state_dict'] if isinstance(
        checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
    if isinstance(state_dict, dict) and all(k.startswith('model.') for k in state_dict.keys()):
        state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(
            f'[warn] Missing keys while loading checkpoint ({len(missing)} keys)')
    if unexpected:
        print(
            f'[warn] Unexpected keys while loading checkpoint ({len(unexpected)} keys)')

model = model.to(device).eval()


def _load_image_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return img.convert('RGB')


new_saved = 0
failed = 0

todo_records = todo.to_dict('records')
for start in tqdm(range(0, len(todo_records), BATCH_SIZE), desc='ConvNeXt frame embeddings'):
    batch_records = todo_records[start: start + BATCH_SIZE]
    batch_images = []
    batch_out_paths = []

    for rec in batch_records:
        frame_path = Path(rec['frame_path'])
        out_path = Path(rec['embedding_path'])
        try:
            batch_images.append(_load_image_rgb(frame_path))
            batch_out_paths.append(out_path)
        except Exception as exc:
            failed += 1
            print(f'[warn] Failed to load {frame_path}: {exc}')

    if not batch_images:
        continue

    with torch.no_grad():
        inputs = processor(images=batch_images, return_tensors='pt')
        pixel_values = inputs['pixel_values'].to(device)
        outputs = model(pixel_values=pixel_values)
        features = outputs.pooler_output
        if features.ndim == 4:
            features = features.squeeze(-1).squeeze(-1)
        features = features.detach().cpu().numpy().astype(np.float32)

    for out_path, emb in zip(batch_out_paths, features):
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb)
        new_saved += 1

# Save mapping for easy future loading
df[['video_idx', 'Image', 'frame_path', 'embedding_path']].to_csv(
    MANIFEST_PATH, index=False)

existing_count = int(df['embedding_path'].map(
    lambda p: Path(p).exists()).sum())
print(
    f'Done. New saved: {new_saved}, failed loads: {failed}, embeddings present: {existing_count}/{len(df)}\n'
    f'Embeddings dir: {EMBEDDINGS_DIR.resolve()}\n'
    f'Manifest: {MANIFEST_PATH.resolve()}'
)


KeyboardInterrupt: 

In [4]:
def calculate_relative_percentage_difference(train_means, test_means, length_weight=1/3):
    percent_diffs = np.abs(train_means - test_means) / \
        np.maximum(train_means, test_means) * 100
    mean_labels = np.mean(percent_diffs[:-1])
    mean_length = np.mean(percent_diffs[-1])
    total_mean = mean_labels * (1 - length_weight) + \
        mean_length * length_weight

    return total_mean


In [5]:
df


,video_idx,Image,Phase,Grasper,Bipolar,Hook,Scissors,Clipper,Irrigator,SpecimenBag,...,time_til_end_phase_3,time_til_end_phase_4,time_til_end_phase_5,time_til_end_phase_6,time_til_next_phase,split,video_dir,frame_file,frame_path,embedding_path
0,1,1,Preparation,1,0,0,0,0,0,0,...,1470,1568,1641,1733,21,train,video01,video01_000001.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video01/video01_000...
1,1,2,Preparation,1,0,0,0,0,0,0,...,1469,1567,1640,1732,20,train,video01,video01_000002.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video01/video01_000...
2,1,3,Preparation,1,0,0,0,0,0,0,...,1468,1566,1639,1731,19,train,video01,video01_000003.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video01/video01_000...
3,1,4,Preparation,1,0,0,0,0,0,0,...,1467,1565,1638,1730,18,train,video01,video01_000004.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video01/video01_000...
4,1,5,Preparation,0,0,0,0,0,0,0,...,1466,1564,1637,1729,17,train,video01,video01_000005.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video01/video01_000...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184493,80,1720,GallbladderRetraction,1,0,0,0,0,0,0,...,-188,-142,-120,5,5,train,video80,video80_001720.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video80/video80_001...
184494,80,1721,GallbladderRetraction,1,0,0,0,0,0,0,...,-189,-143,-121,4,4,train,video80,video80_001721.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video80/video80_001...
184495,80,1722,GallbladderRetraction,1,0,0,0,0,0,0,...,-190,-144,-122,3,3,train,video80,video80_001722.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video80/video80_001...
184496,80,1723,GallbladderRetraction,1,0,0,0,0,0,0,...,-191,-145,-123,2,2,train,video80,video80_001723.png,/media/HDD1/moritz/AI for Surg/frames_224/vide...,Convenxt_cholec_embeddings/video80/video80_001...


In [6]:
tools = [
    'Grasper',
    'Bipolar',
    'Hook',
    'Scissors',
    'Clipper',
    'Irrigator',
    'SpecimenBag'
]

phases = [
    'P_CalotTriangleDissection',
    'P_CleaningCoagulation',
    'P_ClippingCutting',
    'P_GallbladderDissection',
    'P_GallbladderPackaging',
    'P_GallbladderRetraction',
    'P_Preparation'
]


In [7]:
phase_onehot = pd.get_dummies(df['Phase'], prefix='P')
df = pd.concat([df, phase_onehot], axis=1)


In [8]:
vids = df.groupby('video_idx')[tools + phases].mean()
vids = vids.reset_index()

vids['length'] = df.groupby('video_idx').count()[
    'Image'].reset_index(drop=True)


In [9]:
similarities = []

for i in range(1000):
    np.random.seed(i)
    train_vids = np.random.choice(
        df['video_idx'].unique(), size=60, replace=False)
    test_vids = np.setdiff1d(df['video_idx'].unique(), train_vids)

    train_means = vids[vids['video_idx'].isin(
        train_vids)].iloc[:, 1:].mean(axis=0)
    test_means = vids[vids['video_idx'].isin(
        test_vids)].iloc[:, 1:].mean(axis=0)

    similarity = calculate_relative_percentage_difference(
        train_means, test_means)
    similarities.append(similarity)

print(f'{np.mean(similarities):.2f}')


/tmp/ipykernel_1014149/1712964124.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_length = np.mean(percent_diffs[-1])


10.33


In [ ]:
from sklearn.model_selection import train_test_split
# Approximate stratification by using k-means clustering on the per-video feature means.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Standardize the features (exclude the now present 'video_idx' column at index 0)
scaler = StandardScaler()
scaled_vids = scaler.fit_transform(vids[phases].values)

# Use a small number of clusters (e.g., 4 - 8) since samples are few
# heuristic: 1 cluster per ~10 samples, but at least 2
n_clusters = min(4, max(2, len(vids) // 10))
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)
cluster_labels = kmeans.fit_predict(scaled_vids)

# Use the cluster label as an approximate stratification group
train_vids_strat, test_vids_strat = train_test_split(
    vids['video_idx'],
    test_size=0.25,
    stratify=cluster_labels,
    random_state=0
)


In [ ]:
# Keep vids with a single reset_index; a second reset adds non-probability columns.


In [ ]:
# Check score of silver standard where we know what the labels are
similarities = []

train_means = vids[vids['video_idx'].isin(
    train_vids_strat)].iloc[:, 1:].mean(axis=0)
test_means = vids[vids['video_idx'].isin(
    test_vids_strat)].iloc[:, 1:].mean(axis=0)

similarity = calculate_relative_percentage_difference(train_means, test_means)

print(f'{similarity:.2f}')


In [ ]:
similarities = []

for i in range(1000):
    np.random.seed(i)
    train_vids = np.random.choice(
        df['video_idx'].unique(), size=60, replace=False)
    test_vids = np.setdiff1d(df['video_idx'].unique(), train_vids)

    train_means = vids[vids['video_idx'].isin(
        train_vids)].iloc[:, 1:].mean(axis=0)
    test_means = vids[vids['video_idx'].isin(
        test_vids)].iloc[:, 1:].mean(axis=0)

    similarity = calculate_relative_percentage_difference(
        train_means, test_means)
    similarities.append(similarity)

print(f'{np.mean(similarities):.2f}')


In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

FRAMES_DIR = Path('/media/HDD1/moritz/AI for Surg/frames_224')
DINO_EMBEDDINGS_DIR = Path('DINO_cholec_embeddings')
DINO_MANIFEST_PATH = DINO_EMBEDDINGS_DIR / 'embedding_manifest.csv'
DINO_MODEL_NAME = 'facebook/dinov2-small'
DINO_BATCH_SIZE = 128   # float16 allows larger batches
DINO_NUM_WORKERS = 8    # parallel image loading
DINO_FORCE_RECOMPUTE = True   # overwrite stale embeddings from old model
DINO_FAIL_ON_MISSING = True

if 'df' not in globals():
    df = pd.read_csv('df_all.csv')

if not FRAMES_DIR.exists():
    raise FileNotFoundError(f'Missing frames dir: {FRAMES_DIR}')

DINO_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

video_num = df['video_idx'].astype(int)
frame_num = df['Image'].astype(int)
video_dir = video_num.map(lambda v: f'video{v:02d}')
frame_file = [f'{vd}_{fn:06d}.png' for vd, fn in zip(video_dir, frame_num)]
frame_path = [str(FRAMES_DIR / vd / ff)
              for vd, ff in zip(video_dir, frame_file)]
embedding_path = [
    str(DINO_EMBEDDINGS_DIR / vd / Path(ff).with_suffix('.npy'))
    for vd, ff in zip(video_dir, frame_file)
]

manifest_df = pd.DataFrame({
    'video_idx': video_num,
    'Image': frame_num,
    'frame_path': frame_path,
    'embedding_path': embedding_path,
})

exists_mask = manifest_df['frame_path'].map(lambda p: Path(p).exists())
missing_count = int((~exists_mask).sum())
print(f'Mapped {len(manifest_df)} rows to frame paths. Missing: {missing_count}')
if missing_count > 0 and DINO_FAIL_ON_MISSING:
    missing_examples = manifest_df.loc[~exists_mask, 'frame_path'].head(
        5).tolist()
    raise RuntimeError(
        f'Missing frame files (showing up to 5): {missing_examples}')

pairs = manifest_df.loc[exists_mask, [
    'frame_path', 'embedding_path']].drop_duplicates()
todo = pairs if DINO_FORCE_RECOMPUTE else pairs[~pairs['embedding_path'].map(
    lambda p: Path(p).exists())]
print(
    f'Unique DINO embeddings to write: {len(todo)} (total unique frames: {len(pairs)})')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_fp16 = device.type == 'cuda'
print(f'Using device: {device}, fp16: {use_fp16}')

processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
model = AutoModel.from_pretrained(DINO_MODEL_NAME)
model = model.to(device).eval()
if use_fp16:
    model = model.half()


class FrameDataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            with Image.open(rec['frame_path']) as img:
                pixel_values = self.processor(
                    images=img.convert('RGB'), return_tensors='pt'
                )['pixel_values'].squeeze(0)
            return pixel_values, rec['embedding_path'], True
        except Exception:
            return torch.zeros(3, 224, 224), rec['embedding_path'], False


def collate_fn(batch):
    pixels, paths, valids = zip(*batch)
    return torch.stack(pixels), list(paths), list(valids)


todo_records = todo.to_dict('records')
dataset = FrameDataset(todo_records, processor)
loader = DataLoader(
    dataset,
    batch_size=DINO_BATCH_SIZE,
    num_workers=DINO_NUM_WORKERS,
    pin_memory=device.type == 'cuda',
    prefetch_factor=2 if DINO_NUM_WORKERS > 0 else None,
    collate_fn=collate_fn,
)

new_saved = 0
failed = 0

autocast_ctx = torch.autocast(
    'cuda', dtype=torch.float16) if use_fp16 else torch.autocast('cpu', enabled=False)

with torch.no_grad(), autocast_ctx:
    for pixel_values, out_paths, valids in tqdm(loader, desc='DINO frame embeddings'):
        pixel_values = pixel_values.to(device, non_blocking=True)
        if use_fp16:
            pixel_values = pixel_values.half()

        outputs = model(pixel_values=pixel_values)
        # Mean over all token positions (CLS + patches) — matches pipeline behaviour
        features = outputs.last_hidden_state.mean(dim=1)  # [B, D]
        features = features.float().cpu().numpy().astype(np.float16)

        for feat, out_p, valid in zip(features, out_paths, valids):
            if not valid:
                failed += 1
                continue
            out_path = Path(out_p)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(out_path, feat)   # saved as float16
            new_saved += 1

manifest_df[['video_idx', 'Image', 'frame_path', 'embedding_path']].to_csv(
    DINO_MANIFEST_PATH, index=False)
existing_count = int(manifest_df['embedding_path'].map(
    lambda p: Path(p).exists()).sum())
print(
    f'Done. New saved: {new_saved}, failed loads: {failed}, embeddings present: {existing_count}/{len(manifest_df)}\n'
    f'Embeddings dir: {DINO_EMBEDDINGS_DIR.resolve()}\n'
    f'Manifest: {DINO_MANIFEST_PATH.resolve()}'
)


Mapped 184498 rows to frame paths. Missing: 0
Unique DINO embeddings to write: 184498 (total unique frames: 184498)
Using device: cuda, fp16: True


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINO frame embeddings:   0%|          | 0/1442 [00:00<?, ?it/s]

Done. New saved: 184498, failed loads: 0, embeddings present: 184498/184498
Embeddings dir: /home/moritz/Foundential Model/own experiments/Cholec80/DINO_cholec_embeddings
Manifest: /home/moritz/Foundential Model/own experiments/Cholec80/DINO_cholec_embeddings/embedding_manifest.csv


In [13]:
import random
import json
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def _vid(v):
    s = str(v)
    if s.lower().startswith("video"):
        d = "".join(ch for ch in s if ch.isdigit())
        return f"video{int(d):02d}" if d else s
    return f"video{int(float(s)):02d}"


def _l2(x):
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def load_frame_table(manifest_path="Convenxt_cholec_embeddings/embedding_manifest.csv"):
    df = pd.read_csv(manifest_path)

    if "video_id" not in df.columns:
        if "video_idx" not in df.columns:
            raise ValueError("Need video_id or video_idx")
        df["video_id"] = df["video_idx"].map(_vid)

    if "embedding" not in df.columns:
        if "embedding_path" not in df.columns:
            raise ValueError("Need embedding or embedding_path")
        df["embedding"] = df["embedding_path"]

    out = df[["video_id", "embedding"]].copy()
    out["video_id"] = out["video_id"].map(_vid)
    return out


def load_embeddings(frame_table):
    paths = frame_table["embedding"].tolist()
    x0 = np.load(str(paths[0])).astype(np.float16)
    X = np.empty((len(paths), x0.shape[0]), dtype=np.float16)

    for i, p in enumerate(tqdm(paths, desc="Loading frame embeddings", unit="frame")):
        X[i] = np.load(str(p)).astype(np.float16)

    return _l2(X)


def video_centroids(frame_table, X):
    idx = np.arange(len(frame_table), dtype=np.int64)
    tmp = frame_table.copy()
    tmp["_i"] = idx

    vids = sorted(tmp["video_id"].unique())
    V = []
    nframes = []

    for v in tqdm(vids, desc="Computing video centroids", unit="video"):
        rows = tmp.loc[tmp["video_id"] == v, "_i"].to_numpy()
        c = X[rows].mean(axis=0, keepdims=True).astype(np.float32)
        c = _l2(c)[0]
        V.append(c)
        nframes.append(len(rows))

    V = np.stack(V, axis=0).astype(np.float32)
    nframes = np.asarray(nframes, dtype=np.int64)
    return vids, V, nframes


def kmeans_faiss_gpu(X, k, seed=42, niter=15, gpu_id=0, verbose=True):
    X = np.ascontiguousarray(X, dtype=np.float32)
    n, d = X.shape
    k = int(np.clip(k, 2, n - 1))

    km = faiss.Kmeans(
        d=d, k=k, niter=niter, nredo=3, seed=seed,
        spherical=True, gpu=gpu_id, verbose=verbose, max_points_per_centroid=n
    )
    km.train(X)  # train on ALL frames

    _, I = km.index.search(X, 1)
    return I[:, 0].astype(np.int32), km.centroids


def build_markov_video_features(
    labels_df,
    video_ids,
    instrument_cols=None,
    phase_col="Phase",
    phase_cols=None,
    frame_col=None,
):
    """
    Build per-video Markov-style features from instrument and phase labels.
    """
    df = labels_df.copy()

    if "video_id" not in df.columns:
        if "video_idx" not in df.columns:
            raise ValueError("labels CSV needs video_id or video_idx")
        df["video_id"] = df["video_idx"].map(_vid)
    else:
        df["video_id"] = df["video_id"].map(_vid)

    if frame_col is None:
        for c in ("Image", "frame_idx", "frame", "Frame", "frame_number"):
            if c in df.columns:
                frame_col = c
                break

    if instrument_cols is None:
        default_instruments = [
            "Grasper", "Bipolar", "Hook", "Scissors", "Clipper", "Irrigator", "SpecimenBag"
        ]
        instrument_cols = [c for c in default_instruments if c in df.columns]
    else:
        missing = [c for c in instrument_cols if c not in df.columns]
        if missing:
            raise ValueError(
                f"Missing instrument columns in labels CSV: {missing}")

    if not instrument_cols:
        raise ValueError("No instrument columns found in labels CSV")

    used_phase_col = None
    used_phase_cols = None

    if phase_cols is None and phase_col in df.columns:
        used_phase_col = phase_col
        phase_series = df[phase_col].fillna("UNK").astype(str)
        phase_vocab = sorted(phase_series.unique().tolist())
        phase_to_i = {p: i for i, p in enumerate(phase_vocab)}
        df["_phase_idx"] = phase_series.map(phase_to_i).astype(np.int32)
    else:
        if phase_cols is None:
            phase_cols = sorted([c for c in df.columns if c.startswith("P_")])
        missing = [c for c in phase_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing phase columns in labels CSV: {missing}")
        if not phase_cols:
            raise ValueError(
                "No phase labels found (need Phase or P_* columns)")
        used_phase_cols = list(phase_cols)
        phase_values = df[phase_cols].fillna(0).to_numpy(dtype=np.float32)
        df["_phase_idx"] = np.argmax(phase_values, axis=1).astype(np.int32)

    n_phases = int(df["_phase_idx"].max()) + 1
    n_inst = len(instrument_cols)

    features = []
    for v in tqdm(video_ids, desc="Building Markov features", unit="video"):
        sub = df.loc[df["video_id"] == v]
        if frame_col is not None and frame_col in sub.columns:
            sub = sub.sort_values(frame_col)

        phase_seq = sub["_phase_idx"].to_numpy(dtype=np.int32)
        inst_seq = (
            sub[instrument_cols].fillna(0).to_numpy(dtype=np.float32) > 0.5
        ).astype(np.int32)

        if len(phase_seq) == 0:
            phase_hist = np.zeros(n_phases, dtype=np.float32)
            phase_trans = np.zeros((n_phases, n_phases), dtype=np.float32)
            inst_hist = np.zeros(n_inst, dtype=np.float32)
            inst_trans = np.zeros((n_inst, 4), dtype=np.float32)
        else:
            phase_hist = np.bincount(
                phase_seq, minlength=n_phases).astype(np.float32)
            phase_hist /= np.clip(phase_hist.sum(), 1e-12, None)

            phase_trans = np.zeros((n_phases, n_phases), dtype=np.float32)
            if len(phase_seq) > 1:
                np.add.at(phase_trans, (phase_seq[:-1], phase_seq[1:]), 1)
                row_sums = phase_trans.sum(axis=1, keepdims=True)
                phase_trans = np.divide(
                    phase_trans,
                    row_sums,
                    out=np.zeros_like(phase_trans),
                    where=row_sums > 0,
                )

            inst_hist = inst_seq.mean(axis=0).astype(np.float32)
            inst_trans = np.zeros((n_inst, 4), dtype=np.float32)
            if len(inst_seq) > 1:
                trans_codes = (inst_seq[:-1] * 2 +
                               inst_seq[1:]).astype(np.int32)
                for j in range(n_inst):
                    c = np.bincount(
                        trans_codes[:, j], minlength=4).astype(np.float32)
                    c /= np.clip(c.sum(), 1e-12, None)
                    inst_trans[j] = c

        feat = np.concatenate(
            [
                phase_hist.ravel(),
                phase_trans.ravel(),
                inst_hist.ravel(),
                inst_trans.ravel(),
            ],
            axis=0,
        ).astype(np.float32)
        features.append(feat)

    F = np.stack(features, axis=0).astype(np.float32)
    meta = {
        "instrument_cols": list(instrument_cols),
        "phase_col": used_phase_col,
        "phase_cols": used_phase_cols,
        "n_phases": int(n_phases),
        "frame_col": frame_col,
    }
    return F, meta


def split_by_cluster(
    video_ids,
    labels,
    nframes,
    frame_table,
    test_video_ratio=0.40,
    n_trials=10_000,
    top_k=10,
    seed=42,
    debug=False,
):
    """
    Monte Carlo video split: try n_trials random subsets of videos as test set,
    score each by how close every cluster's test-frame proportion is to
    test_video_ratio, and keep the top_k best.
    """
    rng = np.random.default_rng(seed)
    n = len(video_ids)
    target_test_n = int(round(n * test_video_ratio))
    target_test_n = int(np.clip(target_test_n, 2, n - 1))

    # --- build per-video, per-cluster frame counts -------------------------
    unique_clusters = np.unique(labels)
    n_clusters = len(unique_clusters)
    cluster_to_idx = {c: i for i, c in enumerate(unique_clusters)}

    vid_to_i = {v: i for i, v in enumerate(video_ids)}
    C = np.zeros((n, n_clusters), dtype=np.int64)

    for frame_idx, (vid, cl) in enumerate(
        zip(frame_table["video_id"], labels)
    ):
        vi = vid_to_i[vid]
        ci = cluster_to_idx[cl]
        C[vi, ci] += 1

    total_per_cluster = C.sum(axis=0).astype(np.float64)

    # --- Monte Carlo -------------------------------------------------------
    # keep a sorted list of (score, test_indices) of size top_k
    import heapq

    # use a max-heap (negate scores) so we can efficiently drop the worst
    heap = []  # elements: (-score, trial_index, test_idx_array)
    all_idx = np.arange(n, dtype=np.int64)

    trial_iter = tqdm(range(n_trials), desc="Monte Carlo trials",
                      unit="trial") if debug else range(n_trials)
    for t in trial_iter:
        test_idx = rng.choice(all_idx, size=target_test_n, replace=False)

        test_cluster_frames = C[test_idx].sum(axis=0).astype(np.float64)

        ratios = np.divide(
            test_cluster_frames,
            total_per_cluster,
            out=np.full(n_clusters, test_video_ratio),
            where=total_per_cluster > 0,
        )

        score = np.max(np.abs(ratios - test_video_ratio))

        if len(heap) < top_k:
            heapq.heappush(heap, (-score, t, test_idx.copy()))
        elif score < -heap[0][0]:
            heapq.heapreplace(heap, (-score, t, test_idx.copy()))

    # --- build results sorted best-to-worst --------------------------------
    heap.sort(key=lambda x: -x[0])  # sort by score ascending (best first)

    results = []
    for rank, (neg_score, trial_id, test_idx) in enumerate(heap):
        score = -neg_score
        test_set = set(test_idx.tolist())
        train_ids = [video_ids[i] for i in range(n) if i not in test_set]
        test_ids = [video_ids[i] for i in range(n) if i in test_set]

        cluster_info = []
        for ci, c in enumerate(unique_clusters):
            test_f = int(C[list(test_set), ci].sum())
            total_f = int(total_per_cluster[ci])
            ratio = test_f / total_f if total_f > 0 else 0.0
            cluster_info.append({
                "cluster": int(c),
                "test_frames": test_f,
                "total_frames": total_f,
                "test_ratio": round(ratio, 4),
            })

        if debug:
            print(
                f"\n--- Split rank {rank + 1} (score={score:.4f}, trial={trial_id}) ---")
            for ci in cluster_info:
                print(f"  cluster {ci['cluster']:02d}: "
                      f"test_frames={ci['test_frames']}/{ci['total_frames']}  "
                      f"ratio={ci['test_ratio']:.3f}")

        results.append({
            "rank": rank + 1,
            "score": float(score),
            "train_ids": train_ids,
            "test_ids": test_ids,
            "cluster_info": cluster_info,
        })

    return results


def build_fast_split(
    manifest_path="Convenxt_cholec_embeddings/embedding_manifest.csv",
    output_dir="Convenxt_cholec_embeddings/split_60_40_fast",
    test_video_ratio=0.40,
    test_frame_ratio=0.40,
    k_clusters=None,
    seed=42,
    gpu_id=0,
    vids_df=None,
    labels_df=None,
    use_markov_label_clusters=False,
    labels_csv_path=None,
    label_instrument_cols=None,
    label_phase_col="Phase",
    label_phase_cols=None,
    debug=True,
):
    def _to_video_idx(v):
        s = str(v)
        if s.lower().startswith('video'):
            digits = ''.join(ch for ch in s if ch.isdigit())
            return int(digits)
        return int(float(s))

    def calculate_similarity(train_ids, test_ids):
        if vids_df is None:
            return None

        stats_df = vids_df.copy()
        if "video_idx" not in stats_df.columns:
            if "video_id" in stats_df.columns:
                stats_df["video_idx"] = stats_df["video_id"].map(_to_video_idx)
            else:
                if debug:
                    print(
                        "Skipping similarity score: vids_df needs video_idx or video_id")
                return None

        numeric_cols = stats_df.select_dtypes(
            include=[np.number]).columns.tolist()
        feature_cols = [c for c in numeric_cols if c != "video_idx"]
        if not feature_cols:
            if debug:
                print(
                    "Skipping similarity score: vids_df has no numeric feature columns")
            return None

        per_video = stats_df[["video_idx"] +
                             feature_cols].groupby("video_idx", as_index=False).mean()

        train_vids = np.array([_to_video_idx(v)
                              for v in train_ids], dtype=np.int32)
        test_vids = np.array([_to_video_idx(v)
                             for v in test_ids], dtype=np.int32)
        train_rows = per_video[per_video["video_idx"].isin(train_vids)]
        test_rows = per_video[per_video["video_idx"].isin(test_vids)]
        if train_rows.empty or test_rows.empty:
            if debug:
                print("Skipping similarity score: empty train/test rows after filtering")
            return None

        train_means = train_rows[feature_cols].mean(axis=0, numeric_only=True)
        test_means = test_rows[feature_cols].mean(axis=0, numeric_only=True)
        sim_1_15 = calculate_relative_percentage_difference(
            train_means, test_means, 1/15)
        sim_0 = calculate_relative_percentage_difference(
            train_means, test_means, 0)
        return sim_1_15, sim_0

    if debug:
        print("Loading manifest")
    frame_table = load_frame_table(manifest_path)

    if debug:
        print("Loading embeddings")
    X = load_embeddings(frame_table)

    if debug:
        print("Computing video centroids")
    video_ids, _, nframes = video_centroids(frame_table, X)

    if k_clusters is None:
        k_clusters = int(
            np.clip(round(np.sqrt(len(video_ids))), 2, len(video_ids) - 1))

    markov_meta = None
    cluster_source = "frame_embedding_kmeans"
    if use_markov_label_clusters:
        if labels_df is not None:
            label_source = labels_df.copy()
            if debug:
                print("Using in-memory labels_df for label clustering")
        elif labels_csv_path is not None:
            if debug:
                print(f"Loading labels CSV: {labels_csv_path}")
            label_source = pd.read_csv(labels_csv_path)
        else:
            raise ValueError(
                "Provide labels_df or labels_csv_path when use_markov_label_clusters=True")

        frame_like_cols = {"Image", "frame_idx",
                           "frame", "Frame", "frame_number"}
        has_frame_order = any(
            c in label_source.columns for c in frame_like_cols)
        has_phase_signal = (label_phase_col in label_source.columns) or any(
            c.startswith("P_") for c in label_source.columns
        )

        if has_frame_order and has_phase_signal:
            markov_features, markov_meta = build_markov_video_features(
                labels_df=label_source,
                video_ids=video_ids,
                instrument_cols=label_instrument_cols,
                phase_col=label_phase_col,
                phase_cols=label_phase_cols,
            )
            label_features = markov_features
            cluster_source = "label_markov_kmeans"
            if debug:
                print(
                    f"Using Markov clusters from {len(markov_meta['instrument_cols'])} instruments "
                    f"and {markov_meta['n_phases']} phases"
                )
        else:
            table = label_source.copy()
            if "video_id" not in table.columns:
                if "video_idx" not in table.columns:
                    raise ValueError(
                        "labels_df needs video_idx or video_id for per-video label clustering"
                    )
                table["video_id"] = table["video_idx"].map(_vid)
            else:
                table["video_id"] = table["video_id"].map(_vid)

            numeric_cols = table.select_dtypes(
                include=[np.number]).columns.tolist()
            feature_cols = [c for c in numeric_cols if c != "video_idx"]
            if not feature_cols:
                raise ValueError(
                    "labels_df has no numeric columns for per-video label clustering")

            per_video = table.groupby("video_id", as_index=False)[
                feature_cols].mean()
            per_video = per_video.set_index(
                "video_id").reindex(video_ids).fillna(0.0)
            label_features = per_video[feature_cols].to_numpy(dtype=np.float32)
            markov_meta = {
                "mode": "per_video_numeric_labels",
                "feature_cols": feature_cols,
                "n_features": int(len(feature_cols)),
            }
            cluster_source = "label_table_kmeans"
            if debug:
                print(
                    f"Using per-video label features ({len(feature_cols)} dims) for clustering")

        label_features = _l2(label_features.astype(np.float32))
        if debug:
            print(
                f"Running FAISS GPU k-means on label features, k={k_clusters}, gpu={gpu_id}")
        video_cluster_labels, _ = kmeans_faiss_gpu(
            label_features,
            k=k_clusters,
            seed=seed,
            niter=30,
            gpu_id=gpu_id,
            verbose=debug,
        )
        v_to_cluster = {
            v: int(video_cluster_labels[i]) for i, v in enumerate(video_ids)
        }
        frame_labels = np.array([v_to_cluster[v]
                                for v in frame_table["video_id"]], dtype=np.int32)
    else:
        if debug:
            print(
                f"Running FAISS GPU k-means on frames, k={k_clusters}, gpu={gpu_id}")
        frame_labels, _ = kmeans_faiss_gpu(
            X, k=k_clusters, seed=seed, niter=30, gpu_id=gpu_id, verbose=debug)

    if debug:
        print("Running Monte Carlo split")
    results = split_by_cluster(
        video_ids=video_ids,
        labels=frame_labels,
        nframes=nframes,
        frame_table=frame_table,
        test_video_ratio=test_video_ratio,
        n_trials=1_000_000,
        top_k=20,
        seed=seed,
        debug=debug,
    )

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    all_outputs = []
    vid_to_i = {v: i for i, v in enumerate(video_ids)}
    total_frames = int(nframes.sum())

    for r in results:
        rank = r["rank"]
        train_ids = r["train_ids"]
        test_ids = r["test_ids"]

        test_frames = int(sum(nframes[vid_to_i[v]] for v in test_ids))

        cluster_frame_counts = []
        if debug:
            print(
                f"\n===== Split rank {rank} (MC score={r['score']:.4f}) =====")
            print("Frame counts per cluster:")
        for ci in r["cluster_info"]:
            c = int(ci["cluster"])
            test_f = int(ci["test_frames"])
            total_f = int(ci["total_frames"])
            train_f = total_f - test_f
            cluster_frame_counts.append({
                "cluster": c,
                "train_frames": train_f,
                "test_frames": test_f,
                "total_frames": total_f,
            })
            ratio = (test_f / total_f) if total_f > 0 else 0.0
            if debug:
                print(f"  cluster {c:02d}: train={train_f}, test={test_f}, "
                      f"total={total_f}, test_ratio={ratio:.3f}")

        # --- similarity score ---
        sim, sim_lw0 = calculate_similarity(train_ids, test_ids)
        if debug and sim is not None:
            print(f"  Similarity score: {sim:.2f}")
        if (not debug) and rank <= 20:
            sim_txt = f"{sim:.2f}" if sim is not None else "None"
            print(f"Split {rank}: similarity={sim_txt}")

        metrics = {
            "method": "faiss_gpu_kmeans_monte_carlo",
            "cluster_source": cluster_source,
            "rank": rank,
            "mc_score": r["score"],
            "n_videos": int(len(video_ids)),
            "k_clusters": int(k_clusters),
            "train_video_count": int(len(train_ids)),
            "test_video_count": int(len(test_ids)),
            "test_video_ratio": float(len(test_ids) / len(video_ids)),
            "test_frame_ratio": float(test_frames / total_frames),
            "target_test_video_ratio": float(test_video_ratio),
            "target_test_frame_ratio": float(test_frame_ratio),
            "similarity_score": float(sim) if sim is not None else None,
            "similarity_score_lw0": float(sim_lw0) if sim_lw0 is not None else None,
            "cluster_frame_counts": cluster_frame_counts,
            "markov_label_meta": markov_meta,
            "labels_csv_path": str(labels_csv_path) if labels_csv_path is not None else None,
            "seed": int(seed),
            "gpu_id": int(gpu_id),
        }

        # save each split to its own subdirectory
        subdir = out / f"split_rank_{rank:02d}"
        subdir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({"video_id": train_ids}).to_csv(
            subdir / "train_video_ids.csv", index=False)
        pd.DataFrame({"video_id": test_ids}).to_csv(
            subdir / "test_video_ids.csv", index=False)
        with open(subdir / "split_metrics.json", "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        all_outputs.append({
            "train_video_ids": train_ids,
            "test_video_ids": test_ids,
            "metrics": metrics,
        })

    return all_outputs


all_run_results = []
best_similarity_per_run = []
best_similarity_lw0_per_run = []

for k in [20] * 1:
    res = build_fast_split(
        manifest_path="DINO_cholec_embeddings/embedding_manifest.csv",
        output_dir="Convenxt_cholec_embeddings/split_60_40_fast",
        test_video_ratio=0.2,
        test_frame_ratio=0.2,
        seed=random.randint(0, 9999999),
        gpu_id=0,
        vids_df=vids,
        # labels_df=vids,
        k_clusters=k,
        use_markov_label_clusters=False,
        debug=True,
    )

    all_run_results.append(res)

    sims = [
        r["metrics"].get("similarity_score")
        for r in res
        if r["metrics"].get("similarity_score") is not None
    ]
    sims_lw0 = [
        r["metrics"].get("similarity_score_lw0")
        for r in res
        if r["metrics"].get("similarity_score_lw0") is not None
    ]
    best_similarity_per_run.append(sims[0] if sims else None)
    best_similarity_lw0_per_run.append(sims_lw0[0] if sims_lw0 else None)

print(best_similarity_per_run)
print(best_similarity_lw0_per_run)


Loading manifest
Loading embeddings


Loading frame embeddings:   0%|          | 0/184498 [00:00<?, ?frame/s]

Computing video centroids


Computing video centroids:   0%|          | 0/80 [00:00<?, ?video/s]

Running FAISS GPU k-means on frames, k=20, gpu=0
Clustering 184498 points in 384D to 20 clusters, redo 3 times, 30 iterations
  Preprocessing in 0.04 s
Outer iteration 0 / 3
  Iteration 29 (6.95 s, search 6.64 s): objective=160646 imbalance=1.108 nsplit=0       
Objective improved: keep new clusters
Outer iteration 1 / 3
  Iteration 29 (13.03 s, search 12.43 s): objective=160570 imbalance=1.103 nsplit=0       
Outer iteration 2 / 3
  Iteration 29 (19.11 s, search 18.22 s): objective=160656 imbalance=1.114 nsplit=0       
Objective improved: keep new clusters
Running Monte Carlo split


Monte Carlo trials:   0%|          | 0/1000000 [00:00<?, ?trial/s]


--- Split rank 1 (score=0.0421, trial=819120) ---
  cluster 00: test_frames=2008/11055  ratio=0.182
  cluster 01: test_frames=2492/11057  ratio=0.225
  cluster 02: test_frames=321/1796  ratio=0.179
  cluster 03: test_frames=2898/13501  ratio=0.215
  cluster 04: test_frames=819/4659  ratio=0.176
  cluster 05: test_frames=1437/9101  ratio=0.158
  cluster 06: test_frames=2573/12720  ratio=0.202
  cluster 07: test_frames=2323/10355  ratio=0.224
  cluster 08: test_frames=898/5124  ratio=0.175
  cluster 09: test_frames=1221/7442  ratio=0.164
  cluster 10: test_frames=1261/7563  ratio=0.167
  cluster 11: test_frames=2248/10752  ratio=0.209
  cluster 12: test_frames=2971/14070  ratio=0.211
  cluster 13: test_frames=883/5179  ratio=0.171
  cluster 14: test_frames=1826/10107  ratio=0.181
  cluster 15: test_frames=2034/10321  ratio=0.197
  cluster 16: test_frames=1622/7732  ratio=0.210
  cluster 17: test_frames=2064/9057  ratio=0.228
  cluster 18: test_frames=2218/10302  ratio=0.215
  cluster 19

/tmp/ipykernel_1014149/1712964124.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_length = np.mean(percent_diffs[-1])
/tmp/ipykernel_1014149/1712964124.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_length = np.mean(percent_diffs[-1])
/tmp/ipykernel_1014149/1712964124.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_length = np.mean(percent_diffs[-1])
/tmp/ipykernel_1014149/1712964124.py:5: FutureWarning: 

In [ ]:
# ── Foundential datasets: embed + split ──────────────────────────────────────
# Requires the functions from the cell above (load_embeddings, video_centroids,
# kmeans_faiss_gpu, split_by_cluster) to already be defined.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

# ── Config ───────────────────────────────────────────────────────────────────
FOUND_FRAMES_DIR = Path('/media/HDD1/moritz/foundential/Extracted Frames/all')
FOUND_EMB_DIR = Path('/media/HDD1/moritz/Extracted Frames embeddings')
FND_MODEL_NAME = 'facebook/dinov2-base'
FND_BATCH_SIZE = 128
FND_NUM_WORKERS = 8
FND_FORCE_RECOMPUTE = False
FND_TEST_RATIO = 0.2
FND_GPU_ID = 0
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# ── Load model once ───────────────────────────────────────────────────────────
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_use_fp16 = _device.type == 'cuda'
_proc = AutoImageProcessor.from_pretrained(FND_MODEL_NAME)
_dino = AutoModel.from_pretrained(FND_MODEL_NAME).to(_device).eval()
if _use_fp16:
    _dino = _dino.half()
print(f'DINO ready on {_device}')


# ── Dataset / DataLoader ──────────────────────────────────────────────────────
class _FrameDS(Dataset):
    def __init__(self, records, processor):
        self.records, self.processor = records, processor

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            with Image.open(rec['frame_path']) as img:
                pv = self.processor(images=img.convert('RGB'), return_tensors='pt')[
                    'pixel_values'].squeeze(0)
            return pv, rec['embedding_path'], True
        except Exception:
            return torch.zeros(3, 224, 224), rec['embedding_path'], False


def _collate(batch):
    pv, paths, ok = zip(*batch)
    return torch.stack(pv), list(paths), list(ok)


# ── Extraction helper ─────────────────────────────────────────────────────────
def _extract(dataset_dir: Path, emb_dir: Path) -> pd.DataFrame:
    """Embed all frames in dataset_dir/<video>/<frame.*> and return a frame_table."""
    records = []
    for vdir in sorted(dataset_dir.iterdir()):
        if not vdir.is_dir():
            continue
        vid = vdir.name
        for fp in sorted(vdir.iterdir()):
            if fp.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            records.append({
                'video_id':      vid,
                'frame_path':    str(fp),
                'embedding_path': str(emb_dir / vid / fp.with_suffix('.npy').name),
            })

    ft = pd.DataFrame(records)
    if ft.empty:
        return ft

    todo = ft if FND_FORCE_RECOMPUTE else ft[~ft['embedding_path'].map(
        lambda p: Path(p).exists())]
    print(f'  Frames to embed: {len(todo):,} / {len(ft):,}')

    if not todo.empty:
        loader = DataLoader(
            _FrameDS(todo.to_dict('records'), _proc),
            batch_size=FND_BATCH_SIZE, num_workers=FND_NUM_WORKERS,
            pin_memory=_use_fp16,
            prefetch_factor=2 if FND_NUM_WORKERS > 0 else None,
            collate_fn=_collate,
        )
        ac = torch.autocast('cuda', dtype=torch.float16) if _use_fp16 else torch.autocast(
            'cpu', enabled=False)
        saved = failed = 0
        with torch.no_grad(), ac:
            for pv, out_paths, valids in tqdm(loader, desc='  Embedding', leave=False):
                pv = pv.to(_device, non_blocking=True)
                if _use_fp16:
                    pv = pv.half()
                feats = _dino(pixel_values=pv).last_hidden_state.mean(dim=1)
                feats = feats.float().cpu().numpy().astype(np.float16)
                for feat, op, valid in zip(feats, out_paths, valids):
                    if not valid:
                        failed += 1
                        continue
                    p = Path(op)
                    p.parent.mkdir(parents=True, exist_ok=True)
                    np.save(p, feat)
                    saved += 1
        print(f'  Saved {saved:,} embeddings  ({failed} failed)')

    ft = ft.rename(columns={'embedding_path': 'embedding'})
    return ft


# ── Length-only similarity (label means = 0, length_weight = 1) ───────────────
def _length_sim(nframes_arr, vid_to_i, train_ids, test_ids):
    t_len = np.mean([nframes_arr[vid_to_i[v]] for v in train_ids])
    s_len = np.mean([nframes_arr[vid_to_i[v]] for v in test_ids])
    denom = max(t_len, s_len)
    pct = 0.0 if denom == 0 else abs(t_len - s_len) / denom * 100.0
    longer = 'train' if t_len >= s_len else 'test'
    return pct, t_len, s_len, longer


# ── Main loop ─────────────────────────────────────────────────────────────────
FOUND_EMB_DIR.mkdir(parents=True, exist_ok=True)

# Merge VS_Retrosigmoid + VS_Translabyrinthine into a single "VS" dataset
_VS_NAMES = {'VS_Retrosigmoid', 'VS_Translabyrinthine'}
_vs_src_dirs = sorted(d for d in FOUND_FRAMES_DIR.iterdir()
                      if d.is_dir() and d.name in _VS_NAMES)
# tasks: list of (logical_name, [source_dirs])
tasks = [('VS', [_vs_src_dirs[0]]), ('VS', [_vs_src_dirs[1]])
         ] if _vs_src_dirs else []
print(f'\nVS source dirs found: {[d.name for d in _vs_src_dirs]}')
print(
    f'Processing as {len(tasks)} logical dataset(s): {[t[0] for t in tasks]}\n')

# Accumulate all split results — used by the cell below
FOUND_SPLITS = {}
for name, src_dirs in tasks:
    emb_dir = FOUND_EMB_DIR / name
    emb_dir.mkdir(parents=True, exist_ok=True)
    print(f'{"=" * 60}')
    print(f'Dataset: {name}  (sources: {[d.name for d in src_dirs]})')

    # Extract from each source dir into the shared emb_dir, then concatenate
    ft_parts = [_extract(d, emb_dir) for d in src_dirs]
    ft = pd.concat([p for p in ft_parts if not p.empty], ignore_index=True)
    if ft.empty:
        print('  No frames found — skipping.\n')
        continue

    n_vids = ft['video_id'].nunique()
    if n_vids < 4:
        print(f'  Only {n_vids} video(s) — too few to split, skipping.\n')
        continue

    ft[['video_id', 'embedding']].to_csv(
        emb_dir / 'embedding_manifest.csv', index=False)

    X = load_embeddings(ft)
    video_ids, _, nframes = video_centroids(ft, X)
    vid_to_i = {v: i for i, v in enumerate(video_ids)}
    k = n_vids // 2

    print(f'  k-means clusters: {k}  (= {n_vids} videos // 2)')
    frame_labels, _ = kmeans_faiss_gpu(
        X.astype(np.float32), k=k, seed=42, niter=30, gpu_id=FND_GPU_ID, verbose=True
    )

    results = split_by_cluster(
        video_ids=video_ids,
        labels=frame_labels,
        nframes=nframes,
        frame_table=ft,
        test_video_ratio=FND_TEST_RATIO,
        n_trials=10_000_000,
        top_k=1,
        seed=42,
        debug=True,
    )

    best = results[0]
    train_ids = best['train_ids']
    test_ids = best['test_ids']
    sim_pct, train_len, test_len, longer = _length_sim(
        nframes, vid_to_i, train_ids, test_ids)

    print(
        f'  Videos: {n_vids}  |  train: {len(train_ids)}  test: {len(test_ids)}')
    print(f'  Train: {sorted(train_ids)}')
    print(f'  Test:  {sorted(test_ids)}')
    print(f'  Cluster balance score (MC): {best["score"]:.4f}')
    print(f'  Avg frames/video — train: {train_len:.1f}  test: {test_len:.1f}  '
          f'({longer} set is longer, {sim_pct:.2f}% diff)')
    print()

    # ── Save split to disk ────────────────────────────────────────────────────
    split_result = {
        'dataset':    name,
        'n_videos':   n_vids,
        'mc_score':   float(best['score']),
        'train_ids':  sorted(train_ids),
        'test_ids':   sorted(test_ids),
        'train_avg_frames': float(train_len),
        'test_avg_frames':  float(test_len),
        'longer_set': longer,
        'length_pct_diff': float(sim_pct),
    }
    split_json = emb_dir / 'split_result.json'
    with open(split_json, 'w') as f:
        json.dump(split_result, f, indent=2)
    print(f'  Split saved → {split_json}')

    # Store in memory for the cell below
    FOUND_SPLITS[name] = split_result

print(f'\nDone. Splits saved for {len(FOUND_SPLITS)} dataset(s).')
print('Run the next cell to check test-set coverage against the Excel database.')


In [ ]:
# ── Test-set coverage check against Excel database ───────────────────────────
# Requires: FOUND_SPLITS (from cell above) + matching functions from the
# mapping cell (hmzqnkq52sq): _match_atlr, _match_aneurysm, _match_mvd,
# _match_vs, DATASET_CONFIG, VS_HARDCODED, TUMOUR_HARDCODED, xl
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

FOUND_EMB_DIR = Path('/media/HDD1/moritz/Extracted Frames embeddings')

# If FOUND_SPLITS isn't in memory (e.g. cell above wasn't re-run),
# load from the saved JSON files on disk.
if 'FOUND_SPLITS' not in globals() or not FOUND_SPLITS:
    FOUND_SPLITS = {}
    for split_json in sorted(FOUND_EMB_DIR.glob('*/split_result.json')):
        data = json.load(open(split_json))
        FOUND_SPLITS[data['dataset']] = data
    print(f'Loaded {len(FOUND_SPLITS)} split(s) from disk.')

print('=' * 72)
print('Test-set coverage: video ID → Excel database match')
print('=' * 72)

total_test = total_matched = total_unmatched = 0

for ds_name, split in sorted(FOUND_SPLITS.items()):
    test_ids = split['test_ids']
    n_test = len(test_ids)
    total_test += n_test

    print(f'\n{"─" * 72}')
    print(f'Dataset : {ds_name}  —  {n_test} test video(s)')

    cfg = DATASET_CONFIG.get(ds_name)
    if cfg is None:
        print(f'  [WARN] No DATASET_CONFIG entry for {ds_name!r} — skipping.')
        total_unmatched += n_test
        continue

    # ── Tumour: fully hardcoded ───────────────────────────────────────────────
    if cfg['sheet'] is None:
        rows_out = []
        matched = unmatched = 0
        for vid in sorted(test_ids):
            url = TUMOUR_HARDCODED.get(vid)
            if url:
                matched += 1
                rows_out.append(
                    {'test_video': vid, 'status': '✓ (hardcoded)', 'Video Link': url})
            else:
                unmatched += 1
                rows_out.append(
                    {'test_video': vid, 'status': '✗ NO MATCH', 'Video Link': ''})
        total_matched += matched
        total_unmatched += unmatched
        display(pd.DataFrame(rows_out))
        print(
            f'  Matched: {matched}/{n_test}   Unmatched: {unmatched}/{n_test}')
        continue

    # ── All other datasets: look up in Excel ──────────────────────────────────
    df = xl.parse(cfg['sheet'], header=cfg['header'])
    df = df.dropna(how='all')
    df.columns = [str(c).strip() for c in df.columns]

    rows_out = []
    matched = unmatched = 0

    for vid in sorted(test_ids):
        row = cfg['match'](vid, df)
        if row is not None:
            matched += 1
            src = '✓ (hardcoded)' if vid in VS_HARDCODED else '✓'
            meta = {c: row[c] for c in cfg['show'] if c in row.index}
            rows_out.append({'test_video': vid, 'status': src, **meta})
        else:
            unmatched += 1
            rows_out.append({'test_video': vid, 'status': '✗ NO MATCH'})

    total_matched += matched
    total_unmatched += unmatched

    with pd.option_context('display.max_colwidth', 60, 'display.max_rows', 200):
        display(pd.DataFrame(rows_out))
    print(f'  Matched: {matched}/{n_test}   Unmatched: {unmatched}/{n_test}')

print(f'\n{"=" * 72}')
print(f'TEST SET TOTAL  videos  : {total_test}')
print(
    f'                matched : {total_matched}  ({100*total_matched/max(total_test,1):.1f}%)')
print(f'              unmatched : {total_unmatched}')
print('=' * 72)


[["A", "B", "C", "E"],
 ["S", "F", "E", "S"],
 ["A", "D", "E", "E"]]


In [ ]:
# ── Save Excel with "Foundational Model" column + summary sheet ───────────────
# "Foundational Model" column = Y only for TEST-set videos.
import re
import os
import json
import pandas as pd
from pathlib import Path

EXCEL_IN = Path('Non-Pituitary Operative Video Database.xlsx')
EXCEL_OUT = Path(
    'Non-Pituitary Operative Video Database - Foundational Model.xlsx')
FOUND_EMB_DIR = Path('/media/HDD1/moritz/Extracted Frames embeddings')

# ── load test-set splits (from memory or disk) ────────────────────────────────
if 'FOUND_SPLITS' not in globals() or not FOUND_SPLITS:
    FOUND_SPLITS = {}
    for split_json in sorted(FOUND_EMB_DIR.glob('*/split_result.json')):
        data = json.load(open(split_json))
        FOUND_SPLITS[data['dataset']] = data
    print(f'Loaded {len(FOUND_SPLITS)} split(s) from disk.')

# test_ids per dataset  (these are the raw folder names used as video IDs)
test_sets = {ds: set(split['test_ids']) for ds, split in FOUND_SPLITS.items()}

# ── helpers ───────────────────────────────────────────────────────────────────


def _norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


def _vs_prefix(name):
    m = re.match(r'^([A-Za-z0-9-]+)_vestibular', name, re.I)
    return m.group(1) if m else name.split('_')[0]


def _strip_title(f):
    return re.sub(r'_TITLE_\d+$', '', f, flags=re.I)


# ── per-dataset test-norm sets ────────────────────────────────────────────────
atlr_test = test_sets.get('ATLR', set())
aneurysm_test = {_norm(v) for v in test_sets.get('Aneurysm_Clipping', set())}
mvd_test_norms = {_norm(_strip_title(v)) for v in test_sets.get('MVD', set())}
mvd_test_mrns = set()
vs_test_pfx = {_norm(_vs_prefix(v))
               for ds in ('VS_Retrosigmoid', 'VS_Translabyrinthine')
               for v in test_sets.get(ds, set())}
tumour_test = test_sets.get('Tumour_Resections', set())

# build MRN set from MVD test folders (for fallback matching)
for v in test_sets.get('MVD', set()):
    parts = _strip_title(v).split('_')
    if len(parts) >= 4:
        mvd_test_mrns.add(_norm(parts[3]))


def _in_atlr(name):
    return 'Y' if str(name).strip() in atlr_test else ''


def _in_aneurysm(name):
    return 'Y' if _norm(str(name)) in aneurysm_test else ''


def _in_mvd(taxonomy, mrn):
    if _norm(str(taxonomy)) in mvd_test_norms:
        return 'Y'
    mrn_n = _norm(str(mrn))
    if mrn_n and mrn_n in mvd_test_mrns:
        return 'Y'
    return ''


def _in_vs(code):
    return 'Y' if _norm(str(code)) in vs_test_pfx else ''


def _in_tumour(name):
    return 'Y' if str(name) in tumour_test else ''


# ── process every sheet ───────────────────────────────────────────────────────
xl = pd.ExcelFile(EXCEL_IN)
all_sheets_out = {}
summary_rows = []

for sheet in xl.sheet_names:

    if sheet == 'ATLR':
        df_data = xl.parse(sheet, header=1)
        df_data.columns = [str(c).strip() for c in df_data.columns]
        df_data = df_data[df_data['Name'].notna() &
                          ~df_data['Name'].astype(str).str.contains('uploaded', case=False)]
        df_data['Foundational Model'] = df_data['Name'].map(_in_atlr)
        for _, row in df_data[df_data['Foundational Model'] == 'Y'].iterrows():
            summary_rows.append({'Dataset': 'ATLR',
                                 'Name': row['Name'],
                                 'Video Link': row.get('Video Link', '')})
        all_sheets_out[sheet] = df_data

    elif sheet == 'Aneurysm Clipping (real)':
        df_data = xl.parse(sheet, header=1)
        df_data.columns = [str(c).strip() for c in df_data.columns]
        df_data = df_data[df_data['Name'].notna()]
        df_data['Foundational Model'] = df_data['Name'].map(_in_aneurysm)
        for _, row in df_data[df_data['Foundational Model'] == 'Y'].iterrows():
            summary_rows.append({'Dataset': 'Aneurysm_Clipping',
                                 'Name': row['Name'],
                                 'Video Link': row.get('Video Link', '')})
        all_sheets_out[sheet] = df_data

    elif sheet == 'MVD':
        df_data = xl.parse(sheet, header=1)
        df_data.columns = [str(c).strip() for c in df_data.columns]
        df_data = df_data[df_data['Upload Taxonomy'].notna()]
        df_data['Foundational Model'] = [
            _in_mvd(row['Upload Taxonomy'], row.get('MRN', ''))
            for _, row in df_data.iterrows()
        ]
        for _, row in df_data[df_data['Foundational Model'] == 'Y'].iterrows():
            summary_rows.append({'Dataset': 'MVD',
                                 'Name': row['Upload Taxonomy'],
                                 'Video Link': row.get('Link', '')})
        all_sheets_out[sheet] = df_data

    elif sheet == 'Vestibular Schwannoma':
        df_data = xl.parse(sheet, header=1)
        df_data.columns = [str(c).strip() for c in df_data.columns]
        df_data = df_data[df_data['Sortable Code'].notna()]
        df_data['Foundational Model'] = df_data['Sortable Code'].map(_in_vs)
        for _, row in df_data[df_data['Foundational Model'] == 'Y'].iterrows():
            summary_rows.append({'Dataset': 'VS',
                                 'Name': row['Sortable Code'],
                                 'Video Link': row.get('Video Link', '')})
        # inject hardcoded VS test entries missing from spreadsheet
        _vs_hc_pfx = {_norm(code) for code in {'TL-046'}}
        _vs_hc_map = {
            'TL-046': ('TL-046', 'https://www.touchsurgery.com/videos/b572c0b2-721e-42d7-b644-0040a0aabe61')}
        for code, (label, url) in _vs_hc_map.items():
            if _norm(code) in vs_test_pfx:
                summary_rows.append(
                    {'Dataset': 'VS', 'Name': label, 'Video Link': url})
                # mark in df if code matches (won't be in df since not in sheet)
        all_sheets_out[sheet] = df_data

    elif sheet == '5ALA HGG Resections':
        tumour_df = pd.DataFrame([
            {'Name': name, 'Foundational Model': 'Y' if name in tumour_test else '',
             'Video Link': url}
            for name, url in TUMOUR_HARDCODED.items()
        ])
        for _, row in tumour_df[tumour_df['Foundational Model'] == 'Y'].iterrows():
            summary_rows.append({'Dataset': 'Tumour_Resections',
                                 'Name': row['Name'],
                                 'Video Link': row['Video Link']})
        all_sheets_out[sheet] = tumour_df

    else:
        df_data = xl.parse(sheet, header=None)
        if not df_data.empty:
            all_sheets_out[sheet] = df_data

# ── build summary sheet (test set only) ──────────────────────────────────────
summary_df = pd.DataFrame(summary_rows, columns=[
                          'Dataset', 'Name', 'Video Link'])
all_sheets_out['Foundational Model'] = summary_df

# ── write new Excel ───────────────────────────────────────────────────────────
with pd.ExcelWriter(EXCEL_OUT, engine='openpyxl') as writer:
    for sheet_name, df in all_sheets_out.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f'Saved → {EXCEL_OUT}')
print(f'\n"Foundational Model" tab (test set only): {len(summary_df)} videos')
print(summary_df.groupby('Dataset').size().to_string())
print(f'\nTotal: {len(summary_df)}')


In [ ]:
# ── Excel → Folder mapping: verify all Foundential videos against the database ─
import re
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

EXCEL_PATH = Path('Non-Pituitary Operative Video Database.xlsx')
FOUND_FRAMES = Path('/media/HDD1/moritz/foundential/Extracted Frames')

# ── hardcoded entries not (yet) in the Excel ─────────────────────────────────

# VS entries that are missing from the spreadsheet — keyed by folder name
VS_HARDCODED = {
    'TL-046_vestibular_schwannoma_resection_29072025_22009616_PG': {
        'Sortable Code': 'TL-046',
        'Consultant':    'PG',
        'Operation Date': '29/07/2025',
        'MRN':           '22009616',
        'Uploaded to SAVANT': 'Y',
        'Video Link':    'https://www.touchsurgery.com/videos/b572c0b2-721e-42d7-b644-0040a0aabe61',
    },
}

# Tumour_Resections — folders in sorted order, URLs supplied by user
_TUMOUR_FOLDERS = [
    '5ALA_001_21052025_21987388_Op01_R-P',
    '5ALA_002_11062025_21995804_Op01_R-O',
    '5ALA_003_25062025_41290002_Op01_R-F',
    '5ALA_004_30072025_22027336_Op01_R-T',
    '5ALA_005_29092025_21077491_Op03_R-TP',
    '5ALA_006_08102025_21636994_Op02_R-T',
    '5ALA_009_15102025_01058077_Op01_L-TP',
    '5ALA_010_09102025_41371211_Op01_L-T',
    '5ALA_011_20102025_22046197_Op01_L-TP',
    '5ALA_012_22102025_22036719_Op01_L-T',
]
_TUMOUR_URLS = [
    'https://www.touchsurgery.com/videos/471ea873-a7ff-4eed-9577-d608bf659d7e',
    'https://www.touchsurgery.com/videos/f89a1ada-c7eb-4548-a518-16a9a6bfe5da',
    'https://www.touchsurgery.com/videos/b8b0bab3-65f5-41a5-bfda-766274e5f015',
    'https://www.touchsurgery.com/videos/b07c8a18-7f58-4173-9415-635359287eef',
    'https://www.touchsurgery.com/videos/6d16bb38-ec46-4521-91f6-b1e20591fe72',
    'https://www.touchsurgery.com/videos/f23b556c-aefa-4d15-bb3f-2f0592cc8d57',
    'https://www.touchsurgery.com/videos/fee307c3-eea4-4e07-ab77-8a84fcf4be40',
    'https://www.touchsurgery.com/videos/3854fec9-b88b-4495-aa46-23c2b92b2af9',
    'https://www.touchsurgery.com/videos/eadf2b67-f563-45b6-bf15-426b48a41a2e',
    'https://www.touchsurgery.com/videos/56499bdd-de9c-4bb7-aa8c-01b4c047a97f',
]
TUMOUR_HARDCODED = dict(zip(_TUMOUR_FOLDERS, _TUMOUR_URLS))

# ── helpers ───────────────────────────────────────────────────────────────────


def _norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


def _vs_prefix(folder_name):
    m = re.match(r'^([A-Za-z0-9-]+)_vestibular', folder_name, re.I)
    return m.group(1) if m else folder_name.split('_')[0]


xl = pd.ExcelFile(EXCEL_PATH)

# ── match functions ───────────────────────────────────────────────────────────


def _match_atlr(folder, df):
    rows = df[df['Name'] == folder.strip()]
    return rows.iloc[0] if len(rows) else None


def _match_aneurysm(folder, df):
    fn = _norm(folder)
    for _, row in df.iterrows():
        if pd.notna(row['Name']) and _norm(row['Name']) == fn:
            return row
    return None


def _match_mvd(folder, df):
    folder = re.sub(r'_TITLE_\d+$', '', folder, flags=re.I)
    fn = _norm(folder)
    for _, row in df.iterrows():
        if _norm(row['Upload Taxonomy']) == fn:
            return row
    for _, row in df.iterrows():
        mrn = _norm(str(row['MRN']))
        if mrn and mrn in fn:
            return row
    return None


def _match_vs(folder, df):
    # 1. check hardcoded overrides first
    if folder in VS_HARDCODED:
        return pd.Series(VS_HARDCODED[folder])
    # 2. look up in spreadsheet
    prefix = _norm(_vs_prefix(folder))
    for _, row in df.iterrows():
        if _norm(str(row['Sortable Code'])) == prefix:
            return row
    return None


# ── per-dataset config ────────────────────────────────────────────────────────
DATASET_CONFIG = {
    'ATLR': {
        'sheet':  'ATLR',
        'header': 1,
        'match':  _match_atlr,
        'show':   ['Name', 'Duration', 'Consent type'],
        'note':   'Only the first 27 (un-numbered) videos are in the folder.',
    },
    'Aneurysm_Clipping': {
        'sheet':  'Aneurysm Clipping (real)',
        'header': 1,
        'match':  _match_aneurysm,
        'show':   ['ID', 'MRN', 'Location', 'Laterality', 'Approach', 'Op Date', 'Title (TS)'],
        'note':   'Folder names are sanitised versions of the "Name" column.',
    },
    'MVD': {
        'sheet':  'MVD',
        'header': 1,
        'match':  _match_mvd,
        'show':   ['Upload Taxonomy', 'Case Identifier', 'MRN', 'Side', 'Op Date', 'Operation Type'],
        'note':   'Some old folders match by MRN; TITLE_* / MOVIE_* placeholders have no entry.',
    },
    'VS_Retrosigmoid': {
        'sheet':  'Vestibular Schwannoma',
        'header': 1,
        'match':  _match_vs,
        'show':   ['Sortable Code', 'Consultant', 'Operation Date', 'MRN', 'Uploaded to SAVANT', 'Video Link'],
        'note':   'TL-046 hardcoded (not yet in spreadsheet).',
    },
    'VS_Translabyrinthine': {
        'sheet':  'Vestibular Schwannoma',
        'header': 1,
        'match':  _match_vs,
        'show':   ['Sortable Code', 'Consultant', 'Operation Date', 'MRN', 'Uploaded to SAVANT', 'Video Link'],
        'note':   'VS sheet covers both RS and TL entries.',
    },
    'Tumour_Resections': {
        'sheet':  None,
        'header': None,
        'match':  None,
        'show':   ['Video Link'],
        'note':   'Hardcoded video links (5ALA HGG Resections sheet currently empty).',
    },
}

# ── run per dataset ───────────────────────────────────────────────────────────
print('=' * 72)
print('Foundential Video Database → Folder Mapping Check')
print('=' * 72)

total_folders = total_matched = total_unmatched = 0

for ds_name, cfg in DATASET_CONFIG.items():
    ds_path = FOUND_FRAMES / ds_name
    if not ds_path.exists():
        print(f'\n[SKIP] {ds_name}: folder not found on disk\n')
        continue

    folders = sorted(f for f in os.listdir(ds_path)
                     if os.path.isdir(ds_path / f))
    n = len(folders)
    total_folders += n

    print(f'\n{"─" * 72}')
    print(f'Dataset : {ds_name}  ({n} video folders)')
    if cfg['note']:
        print(f'Note    : {cfg["note"]}')

    # ── Tumour_Resections: fully hardcoded ────────────────────────────────────
    if cfg['sheet'] is None:
        rows_out = []
        matched = unmatched = 0
        for f in folders:
            url = TUMOUR_HARDCODED.get(f)
            if url:
                matched += 1
                rows_out.append(
                    {'folder': f, 'status': '✓ (hardcoded)', 'Video Link': url})
            else:
                unmatched += 1
                rows_out.append(
                    {'folder': f, 'status': '✗ NO MATCH', 'Video Link': ''})
        total_matched += matched
        total_unmatched += unmatched
        result_df = pd.DataFrame(rows_out)
        with pd.option_context('display.max_colwidth', 80, 'display.max_rows', 200):
            display(result_df)
        print(f'  Matched: {matched}/{n}   Unmatched: {unmatched}/{n}')
        continue

    # ── all other datasets: look up in Excel ──────────────────────────────────
    df = xl.parse(cfg['sheet'], header=cfg['header'])
    df = df.dropna(how='all')
    df.columns = [str(c).strip() for c in df.columns]

    rows_out = []
    matched = unmatched = 0

    for folder in folders:
        row = cfg['match'](folder, df)
        if row is not None:
            matched += 1
            src = '✓ (hardcoded)' if isinstance(
                row, pd.Series) and folder in VS_HARDCODED else '✓'
            meta = {c: row[c] for c in cfg['show'] if c in row.index}
            rows_out.append({'folder': folder, 'status': src, **meta})
        else:
            unmatched += 1
            rows_out.append({'folder': folder, 'status': '✗ NO MATCH'})

    total_matched += matched
    total_unmatched += unmatched

    result_df = pd.DataFrame(rows_out)
    with pd.option_context('display.max_colwidth', 60, 'display.max_rows', 200):
        display(result_df)
    print(f'  Matched: {matched}/{n}   Unmatched: {unmatched}/{n}')

print(f'\n{"=" * 72}')
print(f'TOTAL  folders : {total_folders}')
print(
    f'       matched : {total_matched}  ({100*total_matched/max(total_folders,1):.1f}%)')
print(f'     unmatched : {total_unmatched}')
print('=' * 72)
